# LGN Visual Processing Pipeline

Visualize how the Lateral Geniculate Nucleus (LGN) preprocesses visual stimuli.

## Table of Contents
1. [Visual Pathway Overview](#1-visual-pathway-overview)
2. [Generating Visual Stimuli](#2-generating-visual-stimuli)
3. [Spatial Filtering](#3-spatial-filtering)
4. [Bilinear Interpolation](#4-bilinear-interpolation)
5. [Temporal Filtering](#5-temporal-filtering)
6. [ON vs OFF Cells](#6-on-vs-off-cells)
7. [Complete Pipeline](#7-complete-pipeline)

---

In [ ]:
# Setup
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

# Enable high-DPI display for plots
%matplotlib inline
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11

print(f"JAX version: {jax.__version__}")
print(f"Devices: {jax.devices()}")

In [ ]:
# Configuration - MODIFY THIS PATH to your data directory
DATA_DIR = Path("/path/to/your/data")  # <-- CHANGE THIS
LGN_DATA_PATH = DATA_DIR / "lgn_full_col_cells_3.csv"

# Verify data exists
if not LGN_DATA_PATH.exists():
    print(f"WARNING: LGN data not found at {LGN_DATA_PATH}")
    print("Please update DATA_DIR to point to your data directory.")
    print("Some cells in this notebook will not execute correctly.")
else:
    print(f"LGN data found at: {LGN_DATA_PATH}")

## 1. Visual Pathway Overview

The visual pathway from eye to V1 cortex:

```
Retina -> LGN -> V1
```

### LGN Functions

The **Lateral Geniculate Nucleus** (LGN) acts as a relay station and filter:

1. **Spatial Filtering**: Gaussian convolution extracts local contrast
2. **Temporal Filtering**: Convolution with temporal kernels captures dynamics
3. **Cell Types**: ON cells respond to light increments, OFF cells to decrements

### LGN Cell Properties

| Property | Description |
|----------|-------------|
| Position (x, y) | Location in visual field |
| Spatial Size | Receptive field size |
| Cell Type | ON, OFF, or ON-OFF composite |
| Temporal Kernel | Transfer function for dynamics |

## 2. Generating Visual Stimuli

Let's generate drifting grating stimuli - a standard stimulus for studying visual responses.

In [ ]:
from v1_jax.data.stim_generator import make_drifting_grating

# Generate drifting grating stimulus
key = jax.random.PRNGKey(42)

grating = make_drifting_grating(
    row_size=120,           # Height (before 2x scaling)
    col_size=240,           # Width (before 2x scaling)
    moving_flag=True,       # Drifting vs static
    image_duration=100,     # Duration in ms
    theta=45.0,             # Orientation in degrees
    cpd=0.05,               # Cycles per degree
    temporal_f=2.0,         # Temporal frequency in Hz
    contrast=1.0,           # Contrast
    key=key,
)

print(f"Grating shape: {grating.shape}")
print(f"  Time steps: {grating.shape[0]}")
print(f"  Height: {grating.shape[1]}")
print(f"  Width: {grating.shape[2]}")
print(f"  Value range: [{float(grating.min()):.2f}, {float(grating.max()):.2f}]")

In [ ]:
# Visualize stimulus frames
fig, axes = plt.subplots(2, 5, figsize=(15, 6))

for i, ax in enumerate(axes.flat):
    frame_idx = i * 10
    ax.imshow(grating[frame_idx, :, :, 0], cmap='gray', vmin=-1, vmax=1)
    ax.set_title(f't={frame_idx}ms')
    ax.axis('off')

plt.suptitle('Drifting Grating Stimulus (45 deg orientation)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Compare different orientations
orientations = [0, 45, 90, 135]
fig, axes = plt.subplots(1, 4, figsize=(14, 4))

for ax, theta in zip(axes, orientations):
    key, subkey = jax.random.split(key)
    grating_theta = make_drifting_grating(
        row_size=120, col_size=240,
        moving_flag=False,  # Static for comparison
        image_duration=10,
        theta=theta,
        key=subkey,
    )
    ax.imshow(grating_theta[0, :, :, 0], cmap='gray', vmin=-1, vmax=1)
    ax.set_title(f'{theta} degrees')
    ax.axis('off')

plt.suptitle('Grating Orientations', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Spatial Filtering

LGN neurons apply **Gaussian spatial filtering** to extract local contrast. Neurons are grouped by receptive field size for efficient processing.

In [ ]:
from v1_jax.lgn import create_gaussian_kernel_trimmed, gaussian_conv2d

# Visualize Gaussian kernels for different spatial sizes
sigmas = [0.5, 1.0, 2.0, 4.0]
fig, axes = plt.subplots(2, 4, figsize=(14, 6))

for idx, sigma in enumerate(sigmas):
    kernel = create_gaussian_kernel_trimmed(sigma)
    
    # 2D kernel visualization
    axes[0, idx].imshow(kernel, cmap='hot')
    axes[0, idx].set_title(f'sigma={sigma}')
    axes[0, idx].axis('off')
    
    # 1D cross-section
    center = kernel.shape[0] // 2
    axes[1, idx].plot(kernel[center, :], 'b-', linewidth=2)
    axes[1, idx].set_xlabel('Position')
    axes[1, idx].set_ylabel('Weight')
    axes[1, idx].set_title(f'Cross-section')
    axes[1, idx].grid(True, alpha=0.3)

axes[0, 0].set_ylabel('2D Kernel')
plt.suptitle('Gaussian Spatial Filters for Different Receptive Field Sizes', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Apply spatial filtering to a grating frame
frame = grating[50, :, :, 0]  # Single frame

fig, axes = plt.subplots(1, 5, figsize=(16, 4))

axes[0].imshow(frame, cmap='gray', vmin=-1, vmax=1)
axes[0].set_title('Original')
axes[0].axis('off')

for idx, sigma in enumerate([0.5, 1.0, 2.0, 4.0]):
    kernel = create_gaussian_kernel_trimmed(sigma)
    filtered = gaussian_conv2d(frame[None, ...], kernel)[0]  # Add/remove time dim
    
    axes[idx+1].imshow(filtered, cmap='gray', vmin=-1, vmax=1)
    axes[idx+1].set_title(f'sigma={sigma}')
    axes[idx+1].axis('off')

plt.suptitle('Effect of Spatial Filtering on Grating', fontsize=14)
plt.tight_layout()
plt.show()

print("Observation: Larger sigma = more blurring = lower spatial frequency response")

## 4. Bilinear Interpolation

After spatial filtering, we need to extract responses at each neuron's location. This is done using **bilinear interpolation** between neighboring pixels.

In [ ]:
from v1_jax.lgn import bilinear_select

# Demonstrate bilinear interpolation
# Create a simple test image
test_image = jnp.zeros((1, 10, 10))  # (T=1, H, W)
test_image = test_image.at[0, 4:6, 4:6].set(1.0)  # Bright spot in center

# Sample at various positions
x_coords = jnp.array([4.0, 4.5, 5.0, 5.5, 6.0])  # Across the bright region
y_coords = jnp.array([5.0, 5.0, 5.0, 5.0, 5.0])  # Fixed y

responses = bilinear_select(x_coords, y_coords, test_image)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Show image with sample points
axes[0].imshow(test_image[0], cmap='hot', extent=[0, 10, 10, 0])
axes[0].scatter(x_coords + 0.5, y_coords + 0.5, c='cyan', s=100, marker='x', linewidths=2)
axes[0].set_title('Test Image with Sample Points')
axes[0].set_xlabel('X')
axes[0].set_ylabel('Y')

# Show interpolated responses
axes[1].plot(x_coords, responses[0], 'bo-', markersize=10, linewidth=2)
axes[1].set_xlabel('X position')
axes[1].set_ylabel('Interpolated value')
axes[1].set_title('Bilinear Interpolation Response')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Sample positions and values:")
for x, v in zip(x_coords, responses[0]):
    print(f"  x={float(x):.1f}: {float(v):.3f}")

## 5. Temporal Filtering

After spatial filtering, LGN neurons apply **temporal filtering** using convolution with cell-specific temporal kernels.

In [ ]:
# Visualize temporal kernels (if data is available)
try:
    from v1_jax.lgn import load_lgn_params
    
    lgn_params = load_lgn_params(str(LGN_DATA_PATH))
    
    print(f"Number of LGN neurons: {lgn_params.n_neurons}")
    print(f"Temporal kernel length: {lgn_params.kernel_length} ms")
    
    # Sample some temporal kernels
    fig, axes = plt.subplots(2, 4, figsize=(14, 6))
    time = np.arange(lgn_params.kernel_length)
    
    # Sample neurons from different types
    sample_indices = [0, 100, 500, 1000]
    
    for idx, neuron_idx in enumerate(sample_indices):
        # Dominant kernel
        dom_kernel = lgn_params.dom_temporal_kernels[neuron_idx]
        axes[0, idx].plot(time, dom_kernel, 'b-', linewidth=2)
        axes[0, idx].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
        axes[0, idx].set_title(f'Neuron {neuron_idx}\n{lgn_params.model_id[neuron_idx][:20]}')
        axes[0, idx].set_xlabel('Time (ms)')
        if idx == 0:
            axes[0, idx].set_ylabel('Dominant Kernel')
        axes[0, idx].grid(True, alpha=0.3)
        
        # Non-dominant kernel
        non_dom_kernel = lgn_params.non_dom_temporal_kernels[neuron_idx]
        axes[1, idx].plot(time, non_dom_kernel, 'r-', linewidth=2)
        axes[1, idx].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
        axes[1, idx].set_xlabel('Time (ms)')
        if idx == 0:
            axes[1, idx].set_ylabel('Non-dominant Kernel')
        axes[1, idx].grid(True, alpha=0.3)
    
    plt.suptitle('Temporal Kernels for Different LGN Neurons', fontsize=14)
    plt.tight_layout()
    plt.show()
    
except FileNotFoundError:
    print("LGN data not found. Skipping temporal kernel visualization.")
    print("Update DATA_DIR path to visualize actual temporal kernels.")

In [ ]:
# Demonstrate temporal filtering with synthetic kernel
from v1_jax.lgn import temporal_filter

# Create a biphasic temporal kernel (typical LGN response)
t = np.arange(100)
tau1, tau2 = 10.0, 20.0
biphasic_kernel = (t/tau1) * np.exp(-t/tau1) - 0.5 * (t/tau2) * np.exp(-t/tau2)
biphasic_kernel = biphasic_kernel / np.abs(biphasic_kernel).max()  # Normalize

# Create a step input signal
signal = np.zeros(200)
signal[50:100] = 1.0  # Step on at t=50, off at t=100

# Apply temporal filtering
kernel_jax = jnp.array(biphasic_kernel[:, None])  # (kernel_len, n_neurons=1)
signal_jax = jnp.array(signal[:, None])  # (time, n_neurons=1)

filtered = temporal_filter(signal_jax, kernel_jax)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Temporal kernel
axes[0].plot(t, biphasic_kernel, 'b-', linewidth=2)
axes[0].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Time (ms)')
axes[0].set_ylabel('Amplitude')
axes[0].set_title('Biphasic Temporal Kernel')
axes[0].grid(True, alpha=0.3)

# Input signal
axes[1].plot(signal, 'k-', linewidth=2)
axes[1].set_xlabel('Time (ms)')
axes[1].set_ylabel('Amplitude')
axes[1].set_title('Input Signal (Step)')
axes[1].set_ylim([-0.2, 1.2])
axes[1].grid(True, alpha=0.3)

# Filtered output
axes[2].plot(filtered[:, 0], 'r-', linewidth=2)
axes[2].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
axes[2].set_xlabel('Time (ms)')
axes[2].set_ylabel('Amplitude')
axes[2].set_title('Filtered Output')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Temporal filtering captures the dynamics of visual stimuli.")
print("The biphasic kernel produces transient responses at stimulus onset/offset.")

## 6. ON vs OFF Cells

LGN neurons are classified as:
- **ON cells**: Increase firing rate when light increases
- **OFF cells**: Increase firing rate when light decreases
- **ON-OFF cells**: Respond to both increases and decreases (composite)

In [ ]:
try:
    # Load LGN parameters to analyze cell types
    from collections import Counter
    
    # Count cell types
    model_ids = lgn_params.model_id
    
    on_count = sum(1 for m in model_ids if 'ON' in m and 'OFF' not in m)
    off_count = sum(1 for m in model_ids if 'OFF' in m and 'ON' not in m)
    on_off_count = sum(1 for m in model_ids if 'ON' in m and 'OFF' in m)
    
    print("LGN Cell Type Distribution:")
    print(f"  ON cells:     {on_count:,} ({100*on_count/len(model_ids):.1f}%)")
    print(f"  OFF cells:    {off_count:,} ({100*off_count/len(model_ids):.1f}%)")
    print(f"  ON-OFF cells: {on_off_count:,} ({100*on_off_count/len(model_ids):.1f}%)")
    
    # Visualize cell type distribution
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Pie chart
    sizes = [on_count, off_count, on_off_count]
    labels = ['ON', 'OFF', 'ON-OFF']
    colors = ['red', 'blue', 'purple']
    axes[0].pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
    axes[0].set_title('Cell Type Distribution')
    
    # Spatial distribution of cell types
    on_mask = np.array(['ON' in m and 'OFF' not in m for m in model_ids])
    off_mask = np.array(['OFF' in m and 'ON' not in m for m in model_ids])
    
    axes[1].scatter(lgn_params.x[on_mask], lgn_params.y[on_mask], 
                    s=1, alpha=0.5, c='red', label='ON')
    axes[1].scatter(lgn_params.x[off_mask], lgn_params.y[off_mask], 
                    s=1, alpha=0.5, c='blue', label='OFF')
    axes[1].set_xlabel('X position')
    axes[1].set_ylabel('Y position')
    axes[1].set_title('Spatial Distribution of ON/OFF Cells')
    axes[1].legend()
    
    plt.tight_layout()
    plt.show()
    
except NameError:
    print("LGN parameters not loaded. Skipping cell type analysis.")

In [ ]:
# Demonstrate ON vs OFF response to light step
# Create a simple light step stimulus
time = np.arange(200)
light_step = np.zeros(200)
light_step[50:150] = 1.0  # Light ON from 50-150ms

# ON cell response (increases with light)
on_response = np.clip(light_step * 1.5 + np.random.randn(200) * 0.1, 0, None)

# OFF cell response (increases with darkness)
off_response = np.clip((1 - light_step) * 1.5 + np.random.randn(200) * 0.1, 0, None)

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

# Light stimulus
axes[0].fill_between(time, 0, light_step, alpha=0.3, color='yellow')
axes[0].plot(time, light_step, 'k-', linewidth=2)
axes[0].set_ylabel('Light')
axes[0].set_title('Light Stimulus')
axes[0].set_ylim([-0.1, 1.5])

# ON cell
axes[1].plot(time, on_response, 'r-', linewidth=1.5)
axes[1].set_ylabel('Firing Rate')
axes[1].set_title('ON Cell Response')
axes[1].fill_between(time, 0, on_response, alpha=0.3, color='red')

# OFF cell
axes[2].plot(time, off_response, 'b-', linewidth=1.5)
axes[2].set_ylabel('Firing Rate')
axes[2].set_xlabel('Time (ms)')
axes[2].set_title('OFF Cell Response')
axes[2].fill_between(time, 0, off_response, alpha=0.3, color='blue')

plt.tight_layout()
plt.show()

print("ON cells: fire when light increases")
print("OFF cells: fire when light decreases (darkness increases)")

## 7. Complete Pipeline

Let's run the complete LGN pipeline:

```
movie -> spatial_filtering -> bilinear_select -> temporal_filtering -> firing_rates
```

In [ ]:
try:
    from v1_jax.lgn import LGN
    
    # Create LGN model
    lgn = LGN(lgn_data_path=str(LGN_DATA_PATH))
    print(f"LGN model created with {lgn.n_neurons:,} neurons")
    
    # Generate longer stimulus
    key = jax.random.PRNGKey(0)
    grating_long = make_drifting_grating(
        row_size=120, col_size=240,
        moving_flag=True,
        image_duration=200,
        theta=45.0,
        key=key,
    )
    
    # Remove channel dimension for LGN
    movie = grating_long[:, :, :, 0]
    print(f"Input movie shape: {movie.shape}")
    
    # Run spatial filtering
    print("\nRunning spatial filtering...")
    dom_spatial, non_dom_spatial = lgn.spatial_response(movie)
    print(f"Spatial response shape: {dom_spatial.shape}")
    
    # Run complete pipeline
    print("\nComputing firing rates...")
    firing_rates = lgn.firing_rates_from_spatial(dom_spatial, non_dom_spatial)
    print(f"Firing rates shape: {firing_rates.shape}")
    print(f"Firing rate range: [{float(firing_rates.min()):.2f}, {float(firing_rates.max()):.2f}]")
    
except FileNotFoundError:
    print("LGN data not found. Skipping complete pipeline demo.")
    print("Update DATA_DIR to run the complete pipeline.")

In [ ]:
try:
    # Visualize LGN population response
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Population firing rate heatmap
    im = axes[0, 0].imshow(firing_rates.T, aspect='auto', cmap='hot',
                          extent=[0, firing_rates.shape[0], 0, firing_rates.shape[1]])
    plt.colorbar(im, ax=axes[0, 0], label='Firing Rate')
    axes[0, 0].set_xlabel('Time (ms)')
    axes[0, 0].set_ylabel('LGN Neuron Index')
    axes[0, 0].set_title('LGN Population Response to Drifting Grating')
    
    # Mean population rate over time
    mean_rate = jnp.mean(firing_rates, axis=1)
    axes[0, 1].plot(mean_rate, 'b-', linewidth=2)
    axes[0, 1].set_xlabel('Time (ms)')
    axes[0, 1].set_ylabel('Mean Firing Rate')
    axes[0, 1].set_title('Population Mean Firing Rate')
    axes[0, 1].grid(True, alpha=0.3)
    
    # ON vs OFF cell comparison
    on_mask = jnp.array(['ON' in m and 'OFF' not in m for m in lgn.params.model_id])
    off_mask = jnp.array(['OFF' in m and 'ON' not in m for m in lgn.params.model_id])
    
    mean_on = jnp.mean(firing_rates[:, on_mask], axis=1)
    mean_off = jnp.mean(firing_rates[:, off_mask], axis=1)
    
    axes[1, 0].plot(mean_on, 'r-', linewidth=2, label='ON cells')
    axes[1, 0].plot(mean_off, 'b-', linewidth=2, label='OFF cells')
    axes[1, 0].set_xlabel('Time (ms)')
    axes[1, 0].set_ylabel('Mean Firing Rate')
    axes[1, 0].set_title('ON vs OFF Cell Population Responses')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # Firing rate distribution
    mean_rates = jnp.mean(firing_rates, axis=0)  # Mean rate per neuron
    axes[1, 1].hist(np.array(mean_rates), bins=50, alpha=0.7, edgecolor='black')
    axes[1, 1].set_xlabel('Mean Firing Rate')
    axes[1, 1].set_ylabel('Count')
    axes[1, 1].set_title('Distribution of Mean Firing Rates')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
except NameError:
    print("Firing rates not computed. Skipping visualization.")

In [ ]:
try:
    # Compare responses to different orientations
    orientations = [0, 45, 90, 135]
    responses = {}
    
    key = jax.random.PRNGKey(0)
    for theta in orientations:
        key, subkey = jax.random.split(key)
        grating = make_drifting_grating(
            row_size=120, col_size=240,
            moving_flag=True,
            image_duration=100,
            theta=theta,
            key=subkey,
        )
        movie = grating[:, :, :, 0]
        rates = lgn(movie)
        responses[theta] = jnp.mean(rates, axis=1)  # Mean over neurons
    
    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    colors = ['red', 'green', 'blue', 'purple']
    for (theta, resp), color in zip(responses.items(), colors):
        axes[0].plot(resp, color=color, linewidth=2, label=f'{theta} deg')
    
    axes[0].set_xlabel('Time (ms)')
    axes[0].set_ylabel('Mean Population Rate')
    axes[0].set_title('LGN Response to Different Grating Orientations')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Mean response per orientation
    mean_resp = [float(jnp.mean(responses[theta])) for theta in orientations]
    axes[1].bar(orientations, mean_resp, width=20, color=colors, alpha=0.7)
    axes[1].set_xlabel('Orientation (degrees)')
    axes[1].set_ylabel('Mean Firing Rate')
    axes[1].set_title('Mean LGN Response by Orientation')
    axes[1].set_xticks(orientations)
    axes[1].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
    
    print("Note: LGN responses are relatively orientation-invariant.")
    print("Orientation selectivity emerges in V1, not LGN.")
    
except NameError:
    print("LGN model not available. Skipping orientation comparison.")

---

## Summary

Key takeaways:

1. **LGN Role**: The LGN acts as a relay station between retina and V1, preprocessing visual information

2. **Spatial Filtering**:
   - Gaussian convolution extracts local contrast
   - Neurons are grouped by receptive field size for efficiency
   - Bilinear interpolation samples at neuron locations

3. **Temporal Filtering**:
   - Convolution with temporal kernels captures dynamics
   - Biphasic kernels produce transient responses

4. **Cell Types**:
   - ON cells: respond to light increases
   - OFF cells: respond to light decreases
   - ON-OFF cells: respond to both (composite)

5. **Complete Pipeline**:
   - `movie -> spatial_filter -> temporal_filter -> firing_rates`
   - LGN is relatively orientation-invariant; selectivity emerges in V1

---

## Exercises

1. **Try different grating parameters**: Modify `cpd` (spatial frequency) and `temporal_f` (temporal frequency) to see how LGN responses change.

2. **Compare static vs moving gratings**: Use `moving_flag=False` vs `True` and compare population responses.

3. **Analyze spatial selectivity**: Plot firing rates vs. neuron position to see if there are spatial patterns in the response.

In [ ]:
# Exercise starter: Compare spatial frequencies

# Try different cycles per degree values
cpd_values = [0.02, 0.05, 0.1, 0.2]

print("TODO: Implement spatial frequency comparison")
print("Hint: Generate gratings with different cpd values and compare LGN responses")

---

## Source Files

- `src/v1_jax/lgn/lgn_model.py` - Main LGN class
- `src/v1_jax/lgn/spatial_filter.py` - Gaussian spatial filtering
- `src/v1_jax/lgn/temporal_filter.py` - Temporal convolution
- `src/v1_jax/data/stim_generator.py` - Stimulus generation utilities

## References

- Chen et al., "Data-driven models of visual cortex", Science Advances 2022
- Allen Institute Visual Coding Dataset: https://observatory.brain-map.org/visualcoding/